# Predicting Global Food Loss and Waste

Orchestration notebook. All modelling code lives in `src/`; this notebook
loads it, runs the experiments, and displays the results.

| Module | Responsibility |
|---|---|
| `data_prep` | loading, filtering, feature and target definitions |
| `pipelines` | all eight model configurations (single source of truth) |
| `baselines` | persistence and linear-trend forecasts |
| `validation` | walk-forward time-aware validation |
| `ablations` | year-treatment ablation, hyperparameter searches |
| `visualization` | prediction helpers and plots |

In [ ]:
import sys, pathlib, warnings

REPO_ROOT = pathlib.Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt

from data_prep import load_flw_data, describe_dataset, DENSE_YEAR_MAX, GROUP_COLS, TARGET
from pipelines import MODEL_NAMES, TUNED_PARAMS, build_pipeline
from ablations import year_ablation, save_table
from validation import walk_forward, summarise, save_results
from visualization import plot_raw_data_overview

pd.set_option("display.width", 140)
print("modules loaded")

## 1. Dataset

In [ ]:
flw_data = load_flw_data()
pd.Series(describe_dataset(flw_data)).to_frame("value")

In [ ]:
_ = plot_raw_data_overview(flw_data)
plt.show()

## 2. Does the year feature earn its place?

Each model is fitted three ways on the same random 80/20 split: with `year`
dropped entirely, passed through unscaled, and standardised.

In [ ]:
ablation = year_ablation(flw_data)
save_table(ablation, "year_ablation.csv")
ablation.pivot(index="model", columns="year_mode", values="test_r2").round(4)

## 3. Time-aware validation

A random split lets a model see the same country/commodity/stage in both
halves. Training on years <= Y and testing on years > Y instead measures
whether the models forecast at all. Repeating over several cutoffs gives a
spread rather than a single number, and both naive baselines are scored on
the identical splits.

In [ ]:
validation_data = load_flw_data(year_max=DENSE_YEAR_MAX)
results, predictions = walk_forward(validation_data)
summary = summarise(results)
save_results(results, predictions, summary)
summary

## 3b. Generalising to unseen countries and commodities

The walk-forward split asks whether a model reaches a later year. This asks
the orthogonal question: whether it reaches a country and commodity pair it
was never trained on. Whole pairs are held out, so nothing in the test fold
appeared in training under any year.

Both baselines forecast within a group that by construction never appears in
training, so they collapse to a global constant. Their fallback rate makes
the difficulty of this split explicit.

In [ ]:
from validation import grouped_cv, summarise_grouped

grouped_results, grouped_predictions = grouped_cv(flw_data, verbose=False)
grouped_summary = summarise_grouped(grouped_results)
grouped_results.to_csv(REPO_ROOT / "results" / "grouped_cv_full_results.csv", index=False)
grouped_summary.to_csv(REPO_ROOT / "results" / "grouped_cv_summary.csv")
grouped_summary

## 3c. Does any model actually beat the baseline?

A mean and a standard deviation across folds mixes two things: how much
models differ from each other, and how much folds differ in difficulty. The
second dominates here, so that table cannot answer the question.

Pairing fixes it. Both predictors are scored on identical rows, so the
per-row difference in error cancels fold difficulty and leaves only the
contrast. A model is credited only where the whole bootstrap interval falls
below zero at *every* split.

Squared error and absolute error are reported separately: the first is
driven by large misses, the second by the typical row. Where they disagree,
that disagreement is the result.

In [ ]:
from inference import compare_against_baseline, summarise_comparison, save_comparison

walkforward_comparison = compare_against_baseline(predictions, split_col="cutoff")
walkforward_verdict = summarise_comparison(walkforward_comparison, split_col="cutoff")
save_comparison(walkforward_comparison, walkforward_verdict,
                prefix="paired_comparison_walkforward")

grouped_comparison = compare_against_baseline(grouped_predictions, split_col="fold")
grouped_verdict = summarise_comparison(grouped_comparison, split_col="fold")
save_comparison(grouped_comparison, grouped_verdict, prefix="paired_comparison_grouped")

walkforward_verdict[walkforward_verdict.metric == "absolute_error"].sort_values("mean_difference")

## 4. Predictive capabilities

The flagship model is the random forest. It leads on unseen years
(R² 0.356), leads on unseen country-commodity pairs (0.389), and leads the
random-split comparison (0.645). The decision tree, used for this section
previously, scores −0.084 on unseen years — worse than predicting a
constant, and worse than assuming no change since the last observation.

In [ ]:
from visualization import (
    predict_food_loss,
    food_loss_comparison,
    plot_food_loss_comparison,
    plot_predicted_vs_actual,
    plot_residuals,
    residuals_by_group,
)

X_year, y = flw_data[GROUP_COLS + ["year"]], flw_data[TARGET]

flagship = build_pipeline("Random Forest", year_mode="scaled")
flagship.fit(X_year, y)
print("flagship:", type(flagship[-1]).__name__)

In [ ]:
predict_food_loss(flagship, "China", "Rice", "Storage", year=2021)
predict_food_loss(flagship, "Benin", "Rice", "Storage", year=2024)

### One comparison function, three dimensions

The dimension to vary is an argument; the other two are held fixed.

In [ ]:
_ = plot_food_loss_comparison(flagship, flw_data, vary="food_supply_stage",
                              country="China", commodity="Rice", year=2021)
plt.show()

In [ ]:
_ = plot_food_loss_comparison(flagship, flw_data, vary="commodity",
                              country="Benin", food_supply_stage="Storage", year=2021)
plt.show()

A year with no observations yields predictions alone rather than failing.

In [ ]:
_ = plot_food_loss_comparison(flagship, flw_data, vary="food_supply_stage",
                              country="China", commodity="Rice", year=2030)
plt.show()

### Residual diagnostics

Aggregate scores hide the shape of the errors. These use the held-out
walk-forward predictions, so nothing here is scored on training data.

In [ ]:
rf_walkforward = predictions[predictions.model == "Random Forest"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plot_predicted_vs_actual(predictions, "Random Forest", ax=axes[0])
plot_residuals(predictions, "Random Forest", ax=axes[1])
plt.tight_layout()
fig.savefig(REPO_ROOT / "figures" / "random_forest_residuals.png", dpi=150)
plt.show()

### Where the errors concentrate

An aggregate score can hide a model that does well on well-represented
countries and poorly everywhere else — the question that decides whether it
travels.

In [ ]:
worst_countries = residuals_by_group(predictions, "Random Forest",
                                    group_col="country", top_n=15)
worst_countries

In [ ]:
worst_commodities = residuals_by_group(predictions, "Random Forest",
                                      group_col="commodity", top_n=15)
worst_commodities

### Error against the size of the actual loss

More than four fifths of observations fall below 5% loss, so an aggregate
score is dominated by the common case. Breaking error down by the size of
the observed loss shows whether the model is reliable about the severe
losses an intervention would target.

In [ ]:
from visualization import error_by_loss_band

loss_band_error = error_by_loss_band(predictions, "Random Forest")
loss_band_error.to_csv(REPO_ROOT / "results" / "error_by_loss_band.csv", index=False)
loss_band_error